In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from google.colab import drive
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

# --- SETTINGS ---
BASE_PATH = '/content/drive/MyDrive/nutrient-deficiency'
IMG_SIZE = 224
BATCH_SIZE = 32

crop_map = {'rice_deficiency': 0, 'maize_deficiency': 1, 'coffee_deficiency': 2}
label_map = {'Nitrogen(N)': 0, 'Phosphorus(P)': 1, 'Potassium(K)': 2, 'Healthy': 3}

Mounted at /content/drive


In [3]:
# Create the Master List
data = []
for crop_folder, crop_id in crop_map.items():
    crop_path = os.path.join(BASE_PATH, crop_folder)
    for label_name, label_id in label_map.items():
        f_path = os.path.join(crop_path, label_name)
        if os.path.exists(f_path):
            for img in os.listdir(f_path):
                if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                    data.append([os.path.join(f_path, img), crop_id, label_id])

df = pd.DataFrame(data, columns=['filepath', 'crop_id', 'label_id'])
df.head()

,filepath,crop_id,label_id
0,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
1,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
2,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
3,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0
4,/content/drive/MyDrive/nutrient-deficiency/ric...,0,0


In [4]:
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=42)

In [10]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = img / 255.0
    return img

In [15]:
# Training Dataset Pipeline
train_ds = tf.data.Dataset.from_tensor_slices((train_df['filepath'], train_df['crop_id'], train_df['label_id']))
train_ds = train_ds.map(lambda f, c, l: (
    (load_image(f), tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
), num_parallel_calls=tf.data.AUTOTUNE).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [16]:
# Validation Dataset Pipeline
val_ds = tf.data.Dataset.from_tensor_slices((val_df['filepath'], val_df['crop_id'], val_df['label_id']))
val_ds = val_ds.map(lambda f, c, l: (
    (load_image(f), tf.one_hot(c, 3)),
    tf.one_hot(l, 4)
)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [17]:
# --- BUILD MODEL ---
img_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image_input")
crop_in = layers.Input(shape=(3,), name="crop_input")

# Image Backbone
base = tf.keras.applications.EfficientNetV2S(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = layers.GlobalAveragePooling2D()(base(img_in))
x = layers.Dropout(0.3)(x)

# Fusion
y = layers.Dense(16, activation='relu')(crop_in)
merged = layers.Concatenate()([x, y])
z = layers.Dense(256, activation='relu')(merged)
z = layers.Dropout(0.4)(z)
out = layers.Dense(4, activation='softmax')(z)

model = models.Model(inputs=[img_in, crop_in], outputs=out)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])